# Chapter 3


## Summarizing long documents with Ollama (gemma4:e2b)


### 3.1 Summarizing a document bigger than the context window
Every LLM has a maximum prompt size (the **context window**): instructions and input text must fit within that limit.

When a document is longer than the allowed context, a reliable approach is **MapReduce summarization**:
- split the source text into token-bounded chunks
- summarize each chunk independently (map)
- summarize the collection of chunk summaries (reduce)

This notebook applies that flow to `Moby-Dick.txt` using a local Ollama model (`gemma4:e2b`).


In [1]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

### 3.1.1 Load text and prepare the model
The source text is loaded into a single string (`moby_dick_book`) and then processed through LCEL chains.

In the OpenAI-based version, this step required API key capture with `getpass`. Here we switch to `ChatOllama`, so execution is local and no API key is needed for the model call itself.

The imports in the next cell include:
- `TokenTextSplitter` for token-based chunking
- `RunnableLambda` and `RunnableParallel` for custom and parallel LCEL steps
- `PromptTemplate` and `StrOutputParser` for prompt construction and clean text outputs


In [2]:
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel


In [3]:
!pip install langchain langchain_ollama


In [4]:
llm = ChatOllama(model="gemma4:e2b")


### 3.1.2 Split stage
The split stage transforms one long string into many manageable chunks.

`TokenTextSplitter(chunk_size=3000, chunk_overlap=100)` keeps each piece within a controlled token budget and adds overlap so important context is less likely to be cut between adjacent chunks.

`RunnableLambda` wraps this logic as a chain component and returns a list of dictionaries (`{'chunk': ...}`) that can be consumed by the map stage.


In [5]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [8]:
test = lambda x: [{'chunk': text_chunk,} for text_chunk in TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)]

In [ ]:
test("ciao come stai pirlone")

### 3.1.3 Map stage
The map stage applies the same summarization prompt to every chunk independently.

In LCEL, the pipe operator (`|`) passes output forward step by step: prompt -> model -> parser.
`RunnableParallel` is used so the chunk-level summarizations can be executed in parallel when `.map()` is applied later, which improves throughput on larger inputs.


In [ ]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

### 3.1.4 Reduce stage
After map completes, you have many partial summaries. The reduce stage combines them into one final synthesis.

A `RunnableLambda` first joins all `summary` values into a single `summaries` string. That combined text is then sent through a second prompt and model call to produce the final summary output.

This mirrors classic MapReduce: independent transformation first, aggregation second.


In [ ]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

### 3.1.5 Build the full MapReduce chain
This composition connects all stages into one executable pipeline:
`split -> map -> reduce`.

The key detail is `summarize_map_chain.map()`, which applies the map chain to each split item. Without it, the chain would not fan out over chunk collections.


In [ ]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

### 3.1.6 Execute and inspect results
`map_reduce_chain.invoke(moby_dick_book)` runs the full workflow end to end and returns the final condensed summary as text.

Execution time depends on document length, number of generated chunks, and local model speed. For fast iteration, use a shorter input file; for fuller coverage, use a longer source and expect higher latency.


In [ ]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [ ]:
print(summary)

## 3.2 Summarizing across documents with Ollama (gemma4:e2b)
In this scenario, the input is not one long document but a collection of sources (Wikipedia + local DOCX/PDF/TXT files).

The goal is to merge insights across heterogeneous documents into one coherent output. Compared to section 3.1, the input preparation changes (loader-based `Document` objects), while chain patterns remain similar.


### 3.2.1 Create `Document` objects from Wikipedia
`WikipediaLoader` retrieves one or more pages related to the query and returns them as a list of LangChain `Document` objects.

This is useful because each `Document` keeps text plus metadata, and downstream chains can process all documents uniformly regardless of source.


In [ ]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

### 3.2.2 Load local file sources (DOCX, PDF, TXT)
Different loaders normalize different file formats into the same `Document` abstraction:
- `Docx2txtLoader` for Word files
- `PyPDFLoader` for PDF files
- `TextLoader` for plain text files

Even when reading a single file, each loader returns a list.


In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

### 3.2.3 Merge all sources into one corpus
`all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs` creates one unified document list.

From this point on, the summarization logic does not need to care where each document came from; it only consumes `Document` items.


In [ ]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

### 3.2.4 Why use the refine technique here
For multi-document synthesis, section 3.2 introduces **refine** as an alternative to MapReduce.

Refine builds the final summary progressively: each iteration sees the current summary draft and one new document, then updates the draft. This often preserves document-specific details better than aggressive parallel compression.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate


In [ ]:
!pip install langchain langchain_ollama


In [ ]:
llm = ChatOllama(model="gemma4:e2b")


### 3.2.5 Base summarization prompt
`doc_summary_template` defines the concise style used to summarize raw document text.

In this notebook, the same local model instance (`ChatOllama` + `gemma4:e2b`) is reused for both base summarization and refinement.


In [ ]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

### 3.2.6 Refinement prompt
The refine prompt takes two inputs:
- `current_refined_summary`
- `text` from an additional document

It instructs the model to incorporate only useful new information, otherwise keep the existing summary unchanged. This stabilizes the summary as more documents are processed.


In [ ]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

### 3.2.7 Iterative refinement function
`refine_summary(docs)` loops through documents sequentially, updates the running summary at each step, and stores intermediate inputs for traceability.

This makes the evolution of the summary explicit, which is useful for debugging and evaluation.


In [ ]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

### 3.2.8 Run and inspect final output
`full_summary = refine_summary(all_docs)` executes the multi-source pipeline.

The returned object includes:
- `final_summary`: consolidated result
- `intermediate_steps`: per-document refinement inputs

Printing the full object helps verify how the summary changes while new sources are incorporated.


In [ ]:
full_summary = refine_summary(all_docs)
print(full_summary)